# ELECTRA — Pre-training Text Encoders as Discriminators Rather Than Generators (2020)

_Papers · Transformers & LLMs_

**Replace a few tokens with a small generator's guesses, then train a discriminator to flag which tokens are fake — learning from every token, not just the masked ones.**

---

This notebook is a practice scaffold for the **ELECTRA — Pre-training Text Encoders as Discriminators Rather Than Generators (2020)** lesson. We build the idea up one step at a time — a toy language, a tiny generator and a bigger discriminator, then the replaced-token-detection training step — so you can see how every token contributes a learning signal. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Check the RTD loss by hand on a worked example

ELECTRA's discriminator solves **replaced-token detection (RTD)**: for each token it predicts P(real). The loss is binary cross-entropy over *every* token — `1` for a real token, `0` for one the generator swapped in. Before any training, we reproduce the lesson's worked example to confirm the per-token loss and its mean line up with what we expect.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Sanity-check the worked example: discriminator RTD loss over ALL 4 tokens.
# original "the cat sat down" -> corrupt "the dog sat down": pos2 replaced, rest real.
y  = torch.tensor([1., 0., 1., 1.])              # 1 = real, 0 = replaced
Dt = torch.tensor([0.9, 0.2, 0.8, 0.6])          # discriminator P(real) per token

# Binary cross-entropy per token: -(y log D + (1-y) log(1-D)).
per_tok = -(y * torch.log(Dt) + (1 - y) * torch.log(1 - Dt))

print("worked example RTD loss: per-token", [round(v, 4) for v in per_tok.tolist()],
      " sum", round(per_tok.sum().item(), 4), " mean", round(per_tok.mean().item(), 4))
# per-token [0.1054, 0.2231, 0.2231, 0.5108]  sum 1.0624  mean 0.2656

### Step 2 — Define a toy language and a small Transformer encoder

We need text to corrupt. A tiny grammar (`det noun verb adv`) over a small vocabulary gives us length-4 sentences with real structure. We also define one reusable `Encoder` — a Transformer body with token + position embeddings. We will instantiate it **twice** later: a *small* one for the generator and a *bigger* one for the discriminator, exactly as ELECTRA prescribes.

In [ ]:
# A toy "language": short sentences from a tiny grammar over a small vocab.
V = ["[PAD]", "[MASK]", "the", "a", "cat", "dog", "bird",
     "sat", "ran", "slept", "here", "there", "fast", "slow"]
stoi = {w: i for i, w in enumerate(V)}
PAD = stoi["[PAD]"]
MASK = stoi["[MASK]"]

DET = ["the", "a"]
NOUN = ["cat", "dog", "bird"]
VERB = ["sat", "ran", "slept"]
ADV = ["here", "there", "fast", "slow"]
rng = np.random.RandomState(0)

def sentence():                                  # det noun verb adv  (always length 4)
    return [stoi[rng.choice(DET)], stoi[rng.choice(NOUN)],
            stoi[rng.choice(VERB)], stoi[rng.choice(ADV)]]

def batch(B):
    rows = [sentence() for _ in range(B)]
    return torch.tensor(rows, device=device)     # B x 4

n_tok = 4
vocab = len(V)

# A small Transformer encoder body (shared design; G is small, D is bigger).
class Encoder(nn.Module):
    def __init__(self, d=32, heads=2, layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(n_tok, d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2 * d, batch_first=True)
        self.body = nn.TransformerEncoder(layer, layers)
        self.d = d
    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return self.body(self.emb(x) + self.pos(positions))   # B x n x d

# Mask 15% (>=1 position) of each sentence — the spots the generator will fill in.
def mask_inputs(x, p=0.15):
    masked = x.clone()
    m = torch.zeros_like(x, dtype=torch.bool)
    for b in range(x.shape[0]):
        k = max(1, int(round(p * x.shape[1])))
        pos = torch.tensor(rng.choice(x.shape[1], size=k, replace=False), device=x.device)
        m[b, pos] = True
    masked[m] = MASK
    return masked, m                             # masked ids, mask boolean

### Step 3 — Build the ELECTRA training step and train

This is the heart of the paper. Each step: (a) the **generator** does masked-language-modelling over the masked spots; (b) we **sample** replacement tokens from its softmax and splice them in — detached, so no gradient flows generator→discriminator (it is *not* a GAN); (c) we label each token real/replaced; (d) the **discriminator** scores every token and we take BCE over **all** tokens. The combined loss is `L_mlm + LAMBDA * L_disc`.

In [ ]:
# The ELECTRA training step: generator MLM + discriminator RTD (NOT a GAN).
genc = Encoder(d=16, layers=1).to(device)        # SMALL generator
g_head = nn.Linear(16, vocab).to(device)
denc = Encoder(d=32, layers=2).to(device)        # bigger discriminator
d_head = nn.Linear(32, 1).to(device)

params = (list(genc.parameters()) + list(g_head.parameters()) +
          list(denc.parameters()) + list(d_head.parameters()))
opt = torch.optim.Adam(params, lr=3e-3)
LAMBDA = 50.0

def electra_step(x, all_tokens=True):
    masked, m = mask_inputs(x)

    # (a) generator: MLM over masked positions
    g_logits = g_head(genc(masked))                  # B x n x vocab
    L_mlm = F.cross_entropy(g_logits[m], x[m])

    # (b) sample replacements from the generator softmax; splice into masked spots; DETACH.
    with torch.no_grad():
        probs = F.softmax(g_logits, dim=-1)
        flat = probs.reshape(-1, vocab)
        sampled = torch.multinomial(flat, 1).reshape(x.shape)
    corrupt = x.clone()
    corrupt[m] = sampled[m]                           # masked spots get generator tokens
    corrupt = corrupt.detach()

    # (c) RTD labels: real(1) if corrupt==original else replaced(0)
    is_real = (corrupt == x).float()                 # note: re-sampled original => real

    # (d) discriminator: one-unit sigmoid head, BCE over ALL tokens (or masked-only for the ablation)
    d_logits = d_head(denc(corrupt)).squeeze(-1)     # B x n  (logit of P(real))
    if all_tokens:
        L_disc = F.binary_cross_entropy_with_logits(d_logits, is_real)
    else:                                            # ablation: mimic MLM's 15% coverage
        L_disc = F.binary_cross_entropy_with_logits(d_logits[m], is_real[m])

    loss = L_mlm + LAMBDA * L_disc
    opt.zero_grad()
    loss.backward()
    opt.step()
    return L_mlm.item(), L_disc.item()

print("training tiny ELECTRA (RTD over all tokens):")
for step in range(400):
    lm, ld = electra_step(batch(64))
    if step % 100 == 0:
        print(f"  step {step:3d}  L_mlm {lm:.3f}  L_disc {ld:.3f}")

### Step 4 — Watch the discriminator spot a replaced token

With training done, we run the pipeline once on a fresh sentence: mask it, let the generator fill the gaps, then ask the discriminator for P(real) at each position. The replaced positions should get a noticeably lower P(real) than the untouched ones — the discriminator has learned to flag the generator's fakes.

In [ ]:
# Show the discriminator spotting replaced tokens on a fresh corrupted sentence.
denc.eval()
xb = batch(1)
masked, m = mask_inputs(xb)

with torch.no_grad():
    probs = F.softmax(g_head(genc(masked)), dim=-1)
    samp = torch.multinomial(probs.reshape(-1, vocab), 1).reshape(xb.shape)
    corrupt = xb.clone()
    corrupt[m] = samp[m]
    p_real = torch.sigmoid(d_head(denc(corrupt)).squeeze(-1))[0]

print("original :", [V[i] for i in xb[0].tolist()])
print("corrupt  :", [V[i] for i in corrupt[0].tolist()])
print("P(real)  :", [round(v, 2) for v in p_real.tolist()],
      "   <- low at replaced positions")

## Visualize the data & results

_At equal pretraining compute, does ELECTRA's all-tokens RTD objective beat MLM (and does the all-tokens loss explain it)?_

### Step 1 — Set up the shared toy language and a probe

To compare objectives fairly we re-create the same toy grammar and encoder, plus two helpers. `probe_acc` freezes an encoder and trains a tiny linear classifier on its mean token vector to predict the verb category — a standard way to measure how good the learned representations are. Everything here is self-contained so the comparison cell below can run on its own.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# Toy reproduction: ELECTRA RTD (all tokens) vs MLM vs RTD-restricted-to-masked, at equal steps.
V = ["[PAD]", "[MASK]", "the", "a", "cat", "dog", "bird",
     "sat", "ran", "slept", "here", "there", "fast", "slow"]
stoi = {w: i for i, w in enumerate(V)}
MASK = stoi["[MASK]"]
vocab = len(V)
n_tok = 4

DET = ["the", "a"]
NOUN = ["cat", "dog", "bird"]
VERB = ["sat", "ran", "slept"]
ADV = ["here", "there", "fast", "slow"]
rng = np.random.RandomState(0)

def sent():
    return [stoi[rng.choice(DET)], stoi[rng.choice(NOUN)],
            stoi[rng.choice(VERB)], stoi[rng.choice(ADV)]]

def batch(B):
    return torch.tensor([sent() for _ in range(B)])

class Enc(nn.Module):
    def __init__(s, d=32, layers=2):
        super().__init__()
        s.emb = nn.Embedding(vocab, d)
        s.pos = nn.Embedding(n_tok, d)
        l = nn.TransformerEncoderLayer(d, 2, 2 * d, batch_first=True)
        s.body = nn.TransformerEncoder(l, layers)
        s.d = d
    def forward(s, x):
        p = torch.arange(x.shape[1]).unsqueeze(0)
        return s.body(s.emb(x) + s.pos(p))

def mask_in(x, p=0.15):
    mk = x.clone()
    m = torch.zeros_like(x, dtype=torch.bool)
    for b in range(x.shape[0]):
        k = max(1, int(round(p * x.shape[1])))
        pos = rng.choice(x.shape[1], k, replace=False)
        m[b, pos] = True
    mk[m] = MASK
    return mk, m

# downstream task: predict the verb category (label) from the sentence's mean token vector.
def probe_acc(enc):
    enc.eval()
    X = batch(400)
    with torch.no_grad():
        feats = enc(X).mean(1)
    lab = (X[:, 2] - stoi["sat"])
    clf = nn.Linear(enc.d, 3)
    o = torch.optim.Adam(clf.parameters(), lr=0.05)
    for _ in range(150):
        o.zero_grad()
        F.cross_entropy(clf(feats[:300]), lab[:300]).backward()
        o.step()
    with torch.no_grad():
        preds = clf(feats[300:]).argmax(1)
        return float((preds == lab[300:]).float().mean())

### Step 2 — Define the three training recipes

We pit three pretraining objectives against each other. `train_rtd(..., all_tokens=True)` is full ELECTRA; `train_rtd(..., all_tokens=False)` is the ablation that restricts the RTD loss to only the masked 15% (matching MLM's coverage); and `train_mlm` is the BERT-style baseline. Keeping the encoder size, step count, and data identical isolates the *loss coverage* as the only difference.

In [ ]:
def train_rtd(steps, all_tokens=True):
    torch.manual_seed(0)
    g = Enc(16, 1)
    gh = nn.Linear(16, vocab)
    d = Enc(32, 2)
    dh = nn.Linear(32, 1)
    params = (list(g.parameters()) + list(gh.parameters()) +
              list(d.parameters()) + list(dh.parameters()))
    opt = torch.optim.Adam(params, lr=3e-3)
    for _ in range(steps):
        x = batch(64)
        mk, m = mask_in(x)
        gl = gh(g(mk))
        Lm = F.cross_entropy(gl[m], x[m])
        with torch.no_grad():
            samp = torch.multinomial(F.softmax(gl, -1).reshape(-1, vocab), 1).reshape(x.shape)
        c = x.clone()
        c[m] = samp[m]
        c = c.detach()
        real = (c == x).float()
        dl = dh(d(c)).squeeze(-1)
        if all_tokens:
            Ld = F.binary_cross_entropy_with_logits(dl, real)
        else:
            Ld = F.binary_cross_entropy_with_logits(dl[m], real[m])
        loss = Lm + 50 * Ld
        opt.zero_grad()
        loss.backward()
        opt.step()
    return d

def train_mlm(steps):
    torch.manual_seed(0)
    e = Enc(32, 2)
    h = nn.Linear(32, vocab)
    opt = torch.optim.Adam(list(e.parameters()) + list(h.parameters()), lr=3e-3)
    for _ in range(steps):
        x = batch(64)
        mk, m = mask_in(x)
        Lm = F.cross_entropy(h(e(mk))[m], x[m])
        opt.zero_grad()
        Lm.backward()
        opt.step()
    return e

### Step 3 — Compare probe accuracy at equal pretraining steps

Now we sweep the number of pretraining steps and probe each resulting encoder. The expected pattern: RTD-over-all-tokens beats MLM at every step (it extracts a gradient from every token, roughly 6× more signal per step), while RTD-restricted-to-masked falls back toward MLM — confirming the all-tokens loss, not the architecture, is what makes ELECTRA efficient.

In [ ]:
for steps in [50, 100, 200, 400]:
    acc_rtd_all = round(probe_acc(train_rtd(steps, True)), 3)
    acc_mlm = round(probe_acc(train_mlm(steps)), 3)
    acc_rtd_masked = round(probe_acc(train_rtd(steps, False)), 3)
    print(steps, "RTD-all", acc_rtd_all,
          "MLM", acc_mlm,
          "RTD-masked", acc_rtd_masked)
# RTD over all tokens > MLM at every step; RTD restricted to masked tokens ~ MLM (our small run).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline. You pretrained two encoders on the same toy corpus for the same number of steps —
            one with MLM (BERT-style, learns from the masked 15%) and one with ELECTRA's RTD (a tiny generator
            corrupts the masked spots, the discriminator labels all tokens real/replaced). A frozen linear
            probe on the RTD discriminator beats the same probe on the MLM encoder. What does that demonstrate, and
            what one-line ablation would erase ELECTRA's advantage?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the two probe accuracies at equal pretraining steps; the RTD discriminator wins. — _RTD computes a loss at every token, so per step it extracts ~6× more learning signal than MLM, which only learns from the masked 15%._
- Ablate: restrict the discriminator's RTD loss to only the masked positions (mimicking MLM's 15% coverage). — _If the win came from the architecture rather than the all-tokens loss, restricting the loss wouldn't matter. It does — isolating "loss over all tokens" as the cause._
- Conclude that the all-tokens RTD objective, not a bigger or different network, supplied the efficiency. — _Same encoder size, same steps, same data; only the loss coverage differs._

**Answer:** It demonstrates that the all-tokens RTD objective is more sample-efficient than MLM: at
                 equal compute the discriminator learns a better encoder because every token contributes a
                 real/replaced gradient, versus only the masked 15% under MLM. Restricting the RTD loss to the
                 masked positions only — matching MLM's coverage — is the ablation that erases the advantage,
                 isolating "loss defined over all input tokens" (not the architecture) as the source of the gain.
                 Our CODEVIZ panel shows the RTD discriminator's probe beating the MLM encoder's at equal steps.

</details>

**Problem 2.** Your worked example gave a mean RTD loss of $0.2656$ over 4 tokens with $D = [0.9, 0.2, 0.8, 0.6]$ and
            labels $y = [\text{real}, \text{replaced}, \text{real}, \text{real}]$. Suppose the discriminator gets
            worse at the replaced token: $D_2$ rises from $0.2$ to $0.7$ (it now thinks the fake
            dog is probably real). Does the total loss go up or down, and by how much?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recompute only the changed term. Position 2 is replaced ($y_2=0$), so its loss is $-\log(1-D_2)$. — _For a replaced token the loss penalizes high "probability of real."_
- Old: $-\log(1-0.2) = -\log(0.8) = 0.2231$. New: $-\log(1-0.7) = -\log(0.3) = 1.2040$. — _Saying a fake is 70% real is a confident mistake, so the loss climbs sharply._
- New sum $= 0.1054 + 1.2040 + 0.2231 + 0.5108 = 2.0433$; mean $= 0.5108$ (up from $0.2656$). — _Only the second term changed; it increased by $1.2040 - 0.2231 = 0.9809$._

**Answer:** The loss goes up — total from $1.0624$ to $2.0433$ (mean $0.2656 \to 0.5108$), an
                 increase of $0.9809$ driven entirely by position 2. Calling a replaced token $70\%$ "real" is a
                 confident error on the very signal RTD cares about, so the binary cross-entropy at that position
                 jumps from $0.2231$ to $1.2040$. That gradient is exactly the pressure that teaches the
                 discriminator to flag the generator's fakes.

</details>

**Problem 3.** Ablation — generator size (§3.2). ELECTRA uses a generator only $1/4$–$1/2$ the discriminator's
            size. Predict what happens to the discriminator's learning if you instead make the generator
            equal in size (very strong), and what happens if you make it trivially tiny (very weak).
            Sketch the curve of discriminator quality vs generator size.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Strong generator: its replacements are nearly perfect — almost indistinguishable from the originals. — _If fakes look exactly like real tokens, the real/replaced task is nearly impossible, so $D$ gets a noisy, unlearnable signal and improves little._
- Trivially weak generator: its replacements are obviously wrong (random-looking). — _Spotting them is trivial, so $D$ learns a shortcut and never has to model real language structure — weak features._
- Sweet spot in between: fakes are plausible-but-wrong, so $D$ must understand context to catch them. — _This is the U-shape the paper reports — quality peaks at a generator $1/4$–$1/2$ the discriminator's size._

**Answer:** Discriminator quality is non-monotonic (an inverted-U) in generator size. An
                 equal-size, very strong generator makes the fakes almost perfect, so the discrimination task
                 is too hard and $D$ barely learns; a trivially weak generator makes the fakes obvious, so
                 $D$ learns a shortcut and weak features. Quality peaks with a moderately small generator
                 ($1/4$–$1/2$ of $D$) that produces plausible-but-wrong tokens, forcing $D$ to model context. Our
                 CODEVIZ / notebook ablation reproduces this U-shape on toy text (our small run, not the paper's
                 number).

</details>